# MultiBranch Model - Huấn luyện mô hình đa nhánh

---


In [ ]:
import os
import sys
import numpy as np
from pathlib import Path

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom, RandomBrightness, RandomContrast, RandomTranslation
from keras.optimizers import AdamW
from keras.losses import BinaryFocalCrossentropy
from keras.utils import image_dataset_from_directory, set_random_seed
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report

sys.path.append(os.path.abspath('..'))
from models.MultiBranchModel import 
from src.evaluate import MetricsCallback, plot_training_history, evaluate_model

In [ ]:
IMG_SIZE = 224
EPOCHS = 30
BATCH_SIZE = 32
SEED = 42
OUTPUT_DIR = '../outputs/custom-model'
CACHE = '../cache/custom-model'

set_random_seed(SEED)

# Xóa cache cũ để tránh lỗi (nếu có)
import shutil
if os.path.exists(CACHE):
    shutil.rmtree(CACHE)
os.makedirs(CACHE, exist_ok=True)

## 1. Chuẩn bị dữ liệu

---

In [ ]:
train_dir = '../datasets/LCC_FASD/train'
val_dir = '../datasets/LCC_FASD/val'
test_dir = '../datasets/LCC_FASD/test'

print("Tập Train:")
train_ds = image_dataset_from_directory(
    train_dir, 
    image_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, 
    label_mode='binary',
    shuffle=True,
    seed=SEED
)

print("\nTập Validation:")
val_ds = image_dataset_from_directory(
    val_dir, 
    image_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, 
    label_mode='binary',
    shuffle=False,
)

print("\nTập Test:")
test_ds = image_dataset_from_directory(
    test_dir, 
    image_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE, 
    label_mode='binary',
    shuffle=False,
)

In [ ]:
train_aug = Sequential([
    Rescaling(1./255),                  # Chuẩn hóa pixel
    RandomFlip("horizontal"),           # Lật ngang
    RandomRotation(0.05),               # Xoay ngẫu nhiên 5%
    RandomZoom(0.2),                    # Thu phóng 20%
    RandomBrightness(0.2),              # Thay đổi độ sáng 20%
    RandomContrast(0.2),                # Thay đổi tương phản 20%
    RandomTranslation(0.01, 0.01)       # Dịch chuyển 1%
])

val_aug = Sequential([Rescaling(1./255)])

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache(CACHE + '/train_cache')
train_ds = train_ds.map(lambda x, y: (train_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.cache(CACHE + '/val_cache')
val_ds = val_ds.map(lambda x, y: (val_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

test_ds = test_ds.cache(CACHE + '/test_cache')
test_ds = test_ds.map(lambda x, y: (val_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## 2. Khởi tạo mô hình

---

Mô hình được xây dựng tại [MultiBranchModel](../models/MultiBranchModel.py).

In [ ]:
input_shape = (IMG_SIZE, IMG_SIZE, 3)

model = custome_model(input_shape=input_shape, num_classes=1)

model.summary()

In [ ]:
metrics_callback = MetricsCallback(val_dataset=val_ds, threshold_path=OUTPUT_DIR + '/best_threshold.txt')

callbacks = [
    metrics_callback,

    ModelCheckpoint(
        filepath=OUTPUT_DIR + '/best_model.keras', 
        monitor='val_acer', 
        save_best_only=True, 
        mode='min', 
        verbose=1
    ),

    EarlyStopping(
        monitor='val_acer',
        patience=5,
        mode='min',
        verbose=1,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_acer',
        factor=0.5,
        patience=3,
        mode='min',
        verbose=1
    )
]

model.compile(
    optimizer=AdamW(learning_rate=0.001, weight_decay=1e-5),
    loss=BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)

In [ ]:
def count():
    image_extensions = {'.png'}
    real_count = sum(
        1 for path in Path(train_dir, 'real').rglob('*')
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    spoof_count = sum(
        1 for path in Path(train_dir, 'spoof').rglob('*')
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    return real_count, spoof_count

In [ ]:
real_count, spoof_count = count()
total = real_count + spoof_count

weight_for_0 = (1 / real_count) * (total / 2.0)
weight_for_1 = (1 / spoof_count) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}
print(f"Trọng số bù đắp nhãn: {class_weight}")

## 3. Huấn luyện mô hình

---

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight
)

## 4. Đánh giá kết quả

---

In [ ]:
threshold_best = metrics_callback.best_threshold
print(f"Ngưỡng tốt nhất: {threshold_best}")

y_prob = model.predict(test_ds)
y_pred = (y_prob >= threshold_best).astype(int).flatten()

y_true = np.concatenate([y.numpy() if hasattr(y, 'numpy') else y for x, y in test_ds], axis=0).flatten()

print(classification_report(y_true, y_pred))

In [ ]:
plot_training_history(history.history, output=OUTPUT_DIR + '/training_history.png')

In [ ]:
y_val_prob = model.predict(val_ds)
y_val_true = np.concatenate([y.numpy() if hasattr(y, 'numpy') else y for x, y in val_ds], axis=0).flatten()

In [ ]:
# Đánh giá và vẽ biểu đồ
y_test = (y_true, y_prob)
y_val = (y_val_true, y_val_prob)

evaluate_model(y_test, y_val, threshold_best, output=OUTPUT_DIR + '/evaluation_report.png')